# Breakout Room #4 — Comparar los métodos y medir si funcionan

**Objetivo:**
- Poner a competir **popularidad**, **Apriori**, **filtrado por ítem** y **filtrado por cliente** con un holdout.
- Medir con **hit rate@3**, **precisión@3** y **cobertura**.
- Contestar la pregunta incómoda: ¿acertar la compra es lo mismo que *causarla*? (puente al ARPU y al uplift)

Es el mismo razonamiento de BG/NBD: escondemos los **últimos 90 días**, predecimos con el pasado y comparamos con lo que pasó.
Los cuatro recomendadores ya vienen armados (los construyeron en los breakouts anteriores). Donde dice **`# TODO`** hay algo para que ustedes completen.

In [1]:
!git clone https://github.com/michellesarahi14/marketing-analytics-michelle-santos.git

Cloning into 'marketing-analytics-michelle-santos'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 127 (delta 40), reused 90 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 3.94 MiB | 13.28 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [5]:
# En Google Colab, descomenten la siguiente línea:
!pip install mlxtend

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from mlxtend.frequent_patterns import apriori, association_rules
import os

warnings.filterwarnings("ignore")
VERDE, OSCURO, NARANJA, GRIS = "#00704A", "#11241C", "#C4512F", "#9A988E"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

# Buscamos la carpeta data dinámicamente
DATA_DIR = "data/"
for root, dirs, files in os.walk("."):
    if "orders.csv" in files:
        DATA_DIR = os.path.join(root, "")
        break

print(f"Cargando datos desde: {DATA_DIR}")
orders = pd.read_csv(DATA_DIR + "orders.csv", parse_dates=["order_date"])
items = pd.read_csv(DATA_DIR + "items.csv")
customers = pd.read_csv(DATA_DIR + "customers.csv")

Cargando datos desde: ./marketing-analytics-michelle-santos/Semana_4/data/


## Paso 1 — las dos ventanas (otra vez)

- **Entrenamiento** = todo hasta el corte. De aquí sale la matriz y las reglas. **Nada de lo que pasó después se puede usar.**
- **Holdout** = los 90 días siguientes. De aquí sale la respuesta: qué productos **nuevos** (que el cliente nunca había comprado) probó.

In [6]:
CUTOFF = orders["order_date"].max() - pd.Timedelta(days=90)
train = orders[orders["order_date"] <= CUTOFF]
holdout = orders[orders["order_date"] > CUTOFF]

M = (pd.crosstab(train["customer_id"], train["item"]) > 0).astype(int).reindex(customers["customer_id"], fill_value=0)
productos = list(M.columns); X = M.values.astype(float)
col_agua = productos.index("Agua")

# Lo que el cliente compró en el holdout y NO había comprado antes (el Agua no cuenta: no se recomienda)
H = (pd.crosstab(holdout["customer_id"], holdout["item"]) > 0).astype(int).reindex(index=M.index, columns=productos, fill_value=0).values
NUEVO = H * (1 - X); NUEVO[:, col_agua] = 0
elegibles = NUEVO.sum(axis=1) > 0
print(f"Corte: {CUTOFF.date()}")
print(f"Clientes que probaron algo NUEVO en los 90 días del holdout: {elegibles.sum():,} de {len(M):,} ({elegibles.mean():.0%})")

Corte: 2026-06-01
Clientes que probaron algo NUEVO en los 90 días del holdout: 986 de 2,000 (49%)


## Paso 2 — los cuatro recomendadores (ya resueltos)

Cada uno produce una matriz de **puntajes**: una fila por cliente, una columna por producto.

In [7]:
# 1) Popularidad: a todos lo mismo, lo más comprado
S_pop = np.tile(X.sum(axis=0), (len(M), 1))

# 2) Apriori: suma del lift de las reglas A → B cuyo A el cliente ya compró
canastas = pd.crosstab(train["order_id"], train["item"]).astype(bool)
reglas = association_rules(apriori(canastas, min_support=0.01, use_colnames=True), metric="lift", min_threshold=1.5)
reglas = reglas[(reglas["antecedents"].apply(len) == 1) & (reglas["consequents"].apply(len) == 1)]
L = np.zeros((len(productos), len(productos)))
for _, r in reglas.iterrows():
    L[productos.index(list(r["antecedents"])[0]), productos.index(list(r["consequents"])[0])] = r["lift"]
S_apriori = X @ L

# 3) Filtrado por ítem
S_it = cosine_similarity(X.T); np.fill_diagonal(S_it, 0)
S_item = X @ S_it

# 4) Filtrado por cliente (K = 30 vecinos)
S_us = cosine_similarity(X); np.fill_diagonal(S_us, 0)
S_user = np.zeros_like(X)
for i in range(len(X)):
    vec = np.argsort(-S_us[i])[:30]
    S_user[i] = S_us[i, vec] @ X[vec]

metodos = {"Popularidad": S_pop, "Apriori": S_apriori, "Filtrado por ítem": S_item, "Filtrado por cliente": S_user}
print(f"Reglas de Apriori usadas (lift ≥ 1.5): {len(reglas)}")

Reglas de Apriori usadas (lift ≥ 1.5): 22


## Paso 3 — las métricas

Para cada cliente recomendamos los **3 productos** con mejor puntaje (sin repetir lo que ya compró ni el Agua), y miramos qué pasó en el holdout.

- **Hit rate@3** = de los clientes que probaron algo nuevo, ¿en qué % al menos **una** de las 3 recomendaciones fue algo que probaron?
- **Cobertura** = ¿cuántos productos *distintos* del catálogo recomienda el método a alguien? (un método que recomienda siempre lo mismo es poco útil)

In [9]:
def top_k(scores, K=3):
    s = np.where(X > 0, -np.inf, scores)       # no recomendar lo ya comprado
    s[:, col_agua] = -np.inf                   # ni el Agua
    return np.argsort(-s, axis=1)[:, :K]

def hit_rate_at_k(scores, K=3):
    top = top_k(scores, K)
    # TODO 1: para cada cliente u, ¿al menos una de sus K recomendaciones (top[u]) es un producto NUEVO que probó (NUEVO[u, ...])?
    # Pista: NUEVO[u, top[u]].sum() > 0 ; después promedien solo sobre los clientes 'elegibles'.
    aciertos = NUEVO[np.arange(len(top))[:, None], top].sum(axis=1) > 0
    return aciertos[elegibles].mean()

def precision_at_k(scores, K=3):
    top = top_k(scores, K)
    return np.mean([NUEVO[u, top[u]].sum() / K for u in np.where(elegibles)[0]])

def cobertura(scores, K=3):
    # TODO 2: ¿cuántos productos distintos aparecen en las recomendaciones de todos los clientes?
    # Pista: np.unique(...) sobre top_k(scores, K).
    return len(np.unique(top_k(scores, K)))

res = pd.DataFrame({n: {"hit rate@3": hit_rate_at_k(s), "precisión@3": precision_at_k(s), "cobertura (de 28)": cobertura(s)}
                    for n, s in metodos.items()}).T
res.round(3)

,hit rate@3,precisión@3,cobertura (de 28)
Popularidad,0.347,0.138,13.0
Apriori,0.409,0.152,19.0
Filtrado por ítem,0.586,0.243,28.0
Filtrado por cliente,0.561,0.228,28.0


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
col = [GRIS, GRIS, VERDE, VERDE]
ax[0].barh(res.index[::-1], res["hit rate@3"][::-1] * 100, color=col[::-1]); ax[0].set_xlabel("Hit rate@3 (%)"); ax[0].set_title("¿Acertaron algo nuevo?")
ax[1].barh(res.index[::-1], res["cobertura (de 28)"][::-1], color=col[::-1]); ax[1].set_xlabel("Productos distintos recomendados"); ax[1].set_title("¿Recomiendan variedad?")
plt.tight_layout(); plt.show()

## Paso 4 — ¿le ganamos a lo simple? ¿y vale la pena?

In [10]:
base = res.loc["Popularidad", "hit rate@3"]
res["mejora vs popularidad"] = (res["hit rate@3"] / base - 1)
print((res[["hit rate@3", "mejora vs popularidad"]] * 100).round(1).rename(columns=lambda c: c + " (%)").to_string())

# ARPU del holdout (ingreso por cliente en los 90 días), con los precios del catálogo
precio = items.set_index("item")["precio"]
ingreso = holdout["item"].map(precio).sum()
print(f"\nIngreso del holdout: ${ingreso:,.0f} · ARPU de 90 días: ${ingreso / len(M):.2f}")
print(f"Clientes que probaron algo nuevo SIN que nadie se lo recomendara: {elegibles.mean():.0%} de la base")

                      hit rate@3 (%)  mejora vs popularidad (%)
Popularidad                     34.7                        0.0
Apriori                         40.9                       17.8
Filtrado por ítem               58.6                       69.0
Filtrado por cliente            56.1                       61.7

Ingreso del holdout: $41,914 · ARPU de 90 días: $20.96
Clientes que probaron algo nuevo SIN que nadie se lo recomendara: 49% de la base


## Paso 5 — la pregunta incómoda

El filtrado por ítem acierta mucho más que la popularidad. Pero miren dos cosas del Paso 4:

- Medimos si el cliente **probó** el producto, no si **lo probó porque se lo recomendamos**. En este holdout *nadie recibió ninguna recomendación*: esos "aciertos" ocurrieron solos.
- Para estimar cuánto **ARPU** agrega un recomendador no basta con el hit rate: hace falta saber cuánto habría comprado el cliente *sin* la recomendación.

> Efecto real en ARPU ≈ (tasa de compra **con** recomendación − tasa **sin** recomendación) × precio × margen.
> Es la misma resta de las clases de Uplift: se necesita un grupo de control aleatorio.

1. ¿Qué método elegirían para producción y por qué? Consideren hit rate, variedad (cobertura), costo y explicabilidad.
2. Un directivo dice: "el recomendador acertó el 70% de las compras, ¡subió el ARPU 70%!". ¿Qué le contestarían?

